In [66]:
import numpy as np
import random
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


print("Loading dataset...")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

# Normalize
X = X / 255.0
y = y.astype(int)

# Reduce dataset size
# X = X[:5000]
# y = y[:5000]

num_labeled = 2000

indices = np.arange(len(X))
np.random.shuffle(indices)

labeled_idx = indices[:num_labeled]
unlabeled_idx = indices[num_labeled:]

X_labeled = X[labeled_idx]
y_labeled = y[labeled_idx]

X_unlabeled = X[unlabeled_idx]
y_unlabeled_true = y[unlabeled_idx]  # for evaluation

print(f"Labeled: {len(X_labeled)}, Unlabeled: {len(X_unlabeled)}")

Loading dataset...
Labeled: 2000, Unlabeled: 68000


In [48]:
# print("\nRunning Self-Training...")

# Combine labeled + unlabeled (unlabeled = -1)
y_semi = np.copy(y)
y_semi[unlabeled_idx] = -1

base_clf = GaussianNB()
self_training_model = SelfTrainingClassifier(base_clf)

self_training_model.fit(X, y_semi)

y_pred_self = self_training_model.predict(X_unlabeled)

acc_self = accuracy_score(y_unlabeled_true, y_pred_self)
print("Self-Training Accuracy:", acc_self)

Self-Training Accuracy: 0.49326530612244895


In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

y_semi = np.copy(y)
y_semi[unlabeled_idx] = -1

CONFIDENCE_THRESHOLD = 0.90
MAX_ITER = 100
remaining_mask = y_semi == -1

# Swap GaussianNB for a better-calibrated classifier
base_clf_options = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest":       RandomForestClassifier(n_estimators=100),
}

for clf_name, base_clf in base_clf_options.items():
    print(f"\n===== {clf_name} =====")

    # Reset for each classifier
    y_semi = np.copy(y)
    y_semi[unlabeled_idx] = -1
    remaining_mask = y_semi == -1

    for iteration in range(1, MAX_ITER):
        if remaining_mask.sum() == 0:
            print("All points labeled. Stopping.")
            break

        base_clf.fit(X[~remaining_mask], y_semi[~remaining_mask])

        unlabeled_X = X[remaining_mask]
        probs = base_clf.predict_proba(unlabeled_X)
        max_probs = probs.max(axis=1)
        confident_mask = max_probs >= CONFIDENCE_THRESHOLD

        predicted_labels = base_clf.predict(unlabeled_X)
        y_semi[remaining_mask] = np.where(confident_mask, predicted_labels, -1)

        newly_labeled = confident_mask.sum()
        remaining_mask = y_semi == -1

        y_pred = base_clf.predict(X_unlabeled)
        acc = accuracy_score(y_unlabeled_true, y_pred)

        print(f"  Iter {iteration} | newly labeled={newly_labeled:>5} | "
              f"still unlabeled={remaining_mask.sum():>5} | accuracy={acc:.4f}")

        if newly_labeled == 0:
            print("  No new confident predictions. Stopping early.")
            break


===== Logistic Regression =====
  Iter 1 | newly labeled= 1465 | still unlabeled= 3435 | accuracy=0.6971
  Iter 2 | newly labeled= 1059 | still unlabeled= 2376 | accuracy=0.6933
  Iter 3 | newly labeled=  605 | still unlabeled= 1771 | accuracy=0.6878
  Iter 4 | newly labeled=  356 | still unlabeled= 1415 | accuracy=0.6880
  Iter 5 | newly labeled=  193 | still unlabeled= 1222 | accuracy=0.6869
  Iter 6 | newly labeled=  134 | still unlabeled= 1088 | accuracy=0.6869
  Iter 7 | newly labeled=   87 | still unlabeled= 1001 | accuracy=0.6839
  Iter 8 | newly labeled=   54 | still unlabeled=  947 | accuracy=0.6831
  Iter 9 | newly labeled=   32 | still unlabeled=  915 | accuracy=0.6820
  Iter 10 | newly labeled=   16 | still unlabeled=  899 | accuracy=0.6822
  Iter 11 | newly labeled=   13 | still unlabeled=  886 | accuracy=0.6812
  Iter 12 | newly labeled=   14 | still unlabeled=  872 | accuracy=0.6798
  Iter 13 | newly labeled=    8 | still unlabeled=  864 | accuracy=0.6792
  Iter 14 | ne

In [50]:
print("\nRunning Co-Training...")

# Split features into two views
X1 = X[:, :392]
X2 = X[:, 392:]

X1_labeled = X1[labeled_idx]
X2_labeled = X2[labeled_idx]

X1_unlabeled = X1[unlabeled_idx]
X2_unlabeled = X2[unlabeled_idx]

# Two classifiers
clf1 = DecisionTreeClassifier()
clf2 = LogisticRegression(max_iter=1000)

# Train initial models
clf1.fit(X1_labeled, y_labeled)
clf2.fit(X2_labeled, y_labeled)

# -----------------------------
# Threshold-based Co-Training
# -----------------------------
threshold = 0.9

# Get probabilities
proba1 = clf1.predict_proba(X1_unlabeled)
proba2 = clf2.predict_proba(X2_unlabeled)

# Confidence scores
conf1 = np.max(proba1, axis=1)
conf2 = np.max(proba2, axis=1)

# Predicted labels
pred1 = np.argmax(proba1, axis=1)
pred2 = np.argmax(proba2, axis=1)

# Get indices of high-confidence samples
idx1 = np.where(conf1 >= threshold)[0]
idx2 = np.where(conf2 >= threshold)[0]

print(f"Model1 confident samples: {len(idx1)}")
print(f"Model2 confident samples: {len(idx2)}")

# Model 1 confident samples (used to teach Model 2)
X2_from_1 = X2_unlabeled[idx1]
y1_high = pred1[idx1]

# Model 2 confident samples (used to teach Model 1)
X1_from_2 = X1_unlabeled[idx2]
y2_high = pred2[idx2]

# Model 1 teaches Model 2
clf2.fit(
    np.vstack((X2_labeled, X2_from_1)),
    np.hstack((y_labeled, y1_high))
)

# Model 2 teaches Model 1
clf1.fit(
    np.vstack((X1_labeled, X1_from_2)),
    np.hstack((y_labeled, y2_high))
)

y_pred_cotrain = clf1.predict(X1_unlabeled)

acc_cotrain = accuracy_score(y_unlabeled_true, y_pred_cotrain)
print("Co-Training Accuracy (Threshold):", acc_cotrain)



Running Co-Training...
Model1 confident samples: 4900
Model2 confident samples: 983
Co-Training Accuracy (Threshold): 0.5348979591836734


In [58]:
# print("\nRunning Co-Training (Fixed)...")

# Split features into two views
X1 = X[:, :392]
X2 = X[:, 392:]

X1_lab = X1[labeled_idx].copy()
X2_lab = X2[labeled_idx].copy()
y_lab  = y_labeled.copy()

X1_unlab = X1[unlabeled_idx].copy()
X2_unlab = X2[unlabeled_idx].copy()

# Both well-calibrated classifiers — no DecisionTree
clf1 = LogisticRegression(max_iter=1000)
clf2 = RandomForestClassifier(n_estimators=100)

THRESHOLD = 0.85
MAX_ITER  = 20

for iteration in range(1, MAX_ITER + 1):
    if len(X1_unlab) == 0:
        print("All points labeled. Stopping.")
        break

    clf1.fit(X1_lab, y_lab)
    clf2.fit(X2_lab, y_lab)

    proba1 = clf1.predict_proba(X1_unlab)
    proba2 = clf2.predict_proba(X2_unlab)

    conf1 = proba1.max(axis=1)
    conf2 = proba2.max(axis=1)
    pred1 = proba1.argmax(axis=1)
    pred2 = proba2.argmax(axis=1)

    # Only add points where:
    # 1. Both models are confident
    # 2. Both models AGREE on the label  ← key fix
    both_confident = (conf1 >= THRESHOLD) & (conf2 >= THRESHOLD)
    both_agree     = (pred1 == pred2)
    new_mask       = both_confident & both_agree
    new_idx        = np.where(new_mask)[0]

    if len(new_idx) == 0:
        print(f"Iteration {iteration} | No confident+agreeing samples. Stopping early.")
        break

    # Add to labeled pool
    X1_lab = np.vstack((X1_lab, X1_unlab[new_idx]))
    X2_lab = np.vstack((X2_lab, X2_unlab[new_idx]))
    y_lab  = np.hstack((y_lab,  pred1[new_idx]))  # pred1==pred2 so either works

    # Remove from unlabeled pool
    remaining  = np.setdiff1d(np.arange(len(X1_unlab)), new_idx)
    X1_unlab   = X1_unlab[remaining]
    X2_unlab   = X2_unlab[remaining]

    # Evaluate on remaining unlabeled
    y_pred = clf1.predict(X1_unlab)
    acc    = accuracy_score(y_unlabeled_true[remaining], y_pred)

    # Agreement rate tells you how healthy the co-training is
    agreement_rate = both_agree.mean()

    print(f"Iter {iteration:>2} | newly labeled={len(new_idx):>5} | "
          f"still unlabeled={len(X1_unlab):>5} | "
          f"agreement={agreement_rate:.2f} | accuracy={acc:.4f}")

Iter  1 | newly labeled=    7 | still unlabeled= 4893 | agreement=0.52 | accuracy=0.6548
Iter  2 | newly labeled=   38 | still unlabeled= 4855 | agreement=0.52 | accuracy=0.2709
Iter  3 | newly labeled=   78 | still unlabeled= 4777 | agreement=0.52 | accuracy=0.1470
Iter  4 | newly labeled=  141 | still unlabeled= 4636 | agreement=0.49 | accuracy=0.1068
Iter  5 | newly labeled=  125 | still unlabeled= 4511 | agreement=0.48 | accuracy=0.1082
Iter  6 | newly labeled=  119 | still unlabeled= 4392 | agreement=0.47 | accuracy=0.1091
Iter  7 | newly labeled=   64 | still unlabeled= 4328 | agreement=0.45 | accuracy=0.1051
Iter  8 | newly labeled=   71 | still unlabeled= 4257 | agreement=0.45 | accuracy=0.0933
Iter  9 | newly labeled=   63 | still unlabeled= 4194 | agreement=0.43 | accuracy=0.1013
Iter 10 | newly labeled=   28 | still unlabeled= 4166 | agreement=0.43 | accuracy=0.1023
Iter 11 | newly labeled=   34 | still unlabeled= 4132 | agreement=0.41 | accuracy=0.0978
Iter 12 | newly label

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

print("\nRunning Co-Training (Capped Growth)...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=100, random_state=42)
X_pca = pca.fit_transform(X_scaled)

X1_lab = X_pca[labeled_idx].copy()
X2_lab = X_scaled[labeled_idx].copy()
y_lab  = y_labeled.copy()

X1_unlab = X_pca[unlabeled_idx].copy()
X2_unlab = X_scaled[unlabeled_idx].copy()

clf1 = LogisticRegression(max_iter=1000)
clf2 = SVC(probability=True, kernel='rbf', C=1.0)

THRESHOLD     = 0.70
MAX_ITER      = 30
MIN_AGREEMENT = 0.60
CAP_PER_ITER  = 100   # ← add at most 100 samples per iteration (most confident first)

for iteration in range(1, MAX_ITER + 1):
    if len(X1_unlab) == 0:
        print("All points labeled. Stopping.")
        break

    clf1.fit(X1_lab, y_lab)
    clf2.fit(X2_lab, y_lab)

    proba1 = clf1.predict_proba(X1_unlab)
    proba2 = clf2.predict_proba(X2_unlab)

    conf1 = proba1.max(axis=1)
    conf2 = proba2.max(axis=1)
    pred1 = proba1.argmax(axis=1)
    pred2 = proba2.argmax(axis=1)

    both_agree     = (pred1 == pred2)
    agreement_rate = both_agree.mean()

    if agreement_rate < MIN_AGREEMENT:
        print(f"Iteration {iteration} | Agreement {agreement_rate:.2f} below minimum. Stopping.")
        break

    both_confident = (conf1 >= THRESHOLD) & (conf2 >= THRESHOLD) & both_agree
    candidate_idx  = np.where(both_confident)[0]

    if len(candidate_idx) == 0:
        print(f"Iteration {iteration} | No confident+agreeing samples. Stopping early.")
        break

    # Sort candidates by average confidence, take top CAP_PER_ITER
    avg_conf  = (conf1[candidate_idx] + conf2[candidate_idx]) / 2
    top_order = np.argsort(avg_conf)[::-1]           # descending
    new_idx   = candidate_idx[top_order[:CAP_PER_ITER]]

    X1_lab = np.vstack((X1_lab, X1_unlab[new_idx]))
    X2_lab = np.vstack((X2_lab, X2_unlab[new_idx]))
    y_lab  = np.hstack((y_lab,  pred1[new_idx]))

    remaining = np.setdiff1d(np.arange(len(X1_unlab)), new_idx)
    X1_unlab  = X1_unlab[remaining]
    X2_unlab  = X2_unlab[remaining]

    y_pred = clf1.predict(X1_unlab)
    acc    = accuracy_score(y_unlabeled_true[remaining], y_pred)

    print(f"Iter {iteration:>2} | newly labeled={len(new_idx):>5} | "
          f"still unlabeled={len(X1_unlab):>5} | "
          f"agreement={agreement_rate:.2f} | accuracy={acc:.4f}")


Running Co-Training (Capped Growth)...
Iter  1 | newly labeled=  100 | still unlabeled=67900 | agreement=0.88 | accuracy=0.8683


In [52]:
# print("\nRunning K-Means Clustering...")

kmeans = KMeans(n_clusters=10, random_state=42)
clusters = kmeans.fit_predict(X)

# Assign labels to clusters
cluster_labels = {}

for i in range(10):
    cluster_points = labeled_idx[clusters[labeled_idx] == i]

    if len(cluster_points) == 0:
        continue

    labels = y[cluster_points]
    majority_label = np.bincount(labels).argmax()
    cluster_labels[i] = majority_label

# Predict for unlabeled
y_pred_cluster = []

for idx in unlabeled_idx:
    cluster_id = clusters[idx]
    if cluster_id in cluster_labels:
        y_pred_cluster.append(cluster_labels[cluster_id])
    else:
        y_pred_cluster.append(random.choice(range(10)))

y_pred_cluster = np.array(y_pred_cluster)

acc_cluster = accuracy_score(y_unlabeled_true, y_pred_cluster)
print("Clustering Accuracy:", acc_cluster)

Clustering Accuracy: 0.4904081632653061


In [53]:
print("===== FINAL COMPARISON =====")
print(f"Self-Training Accuracy : {acc_self:.4f}")
print(f"Co-Training Accuracy   : {acc_cotrain:.4f}")
print(f"K-Means Accuracy       : {acc_cluster:.4f}")

===== FINAL COMPARISON =====
Self-Training Accuracy : 0.4933
Co-Training Accuracy   : 0.5349
K-Means Accuracy       : 0.4904
